# NumPy Intermediate — Broadcasting, dtypes & Arithmetic Safety

**Module 2.2 | Edge AI Engineering Bootcamp**

This notebook covers the intermediate concepts that separate beginners from practitioners who write **correct** and **efficient** Edge AI code:

1. Broadcasting — NumPy's rule system for math between different shapes
2. uint8 vs float32 — memory and arithmetic implications
3. The uint8 overflow/wraparound bug (one of the most dangerous in image processing)
4. Safe arithmetic patterns for image processing
5. Aggregation operations (mean, sum, min, max, std)
6. Boolean masking and conditional operations

In [5]:
import numpy as np

---
## 1. Broadcasting — Why (480, 640, 3) + (3,) Works

Broadcasting is NumPy's rule system for doing math between arrays of **different shapes**, without you having to manually resize anything.

### The Broadcasting Rule (memorize this)

NumPy compares shapes **from the rightmost dimension** (right to left). At each position, the dimensions must either be:
- **Equal**, or
- One of them is **1** (or missing entirely)

If this holds for all dimensions, broadcasting succeeds. NumPy mentally "stretches" the smaller array to match, **without actually duplicating data in memory**.

### Visual Example

```
   img:         (480, 640, 3)
   brightness:          (3,)
                          │
                  NumPy compares shapes from
                  RIGHT to LEFT:
   480, 640, 3
             3    ← matches! (3 == 3)
                     brightness gets "stretched" (virtually,
                     not actually copied in memory) across
                     every pixel in the image
```

### Broadcasting in Action: Per-Channel Adjustment

The most common broadcasting use case in image processing: **applying different adjustments to each color channel**.

### The Setup

```
Image:       (4, 4, 3)  — 4×4 pixels, 3 channels (R, G, B)
Adjustment:  (3,)       — one value per channel: [10, 5, 0]
```

### What Happens

NumPy broadcasts the `(3,)` array across every pixel:

```
Pixel [0,0] = [R, G, B] + [10, 5, 0] = [R+10, G+5, B+0]
Pixel [0,1] = [R, G, B] + [10, 5, 0] = [R+10, G+5, B+0]
Pixel [0,2] = [R, G, B] + [10, 5, 0] = [R+10, G+5, B+0]
...same for every pixel in the image
```

### Why No Copy Happens

NumPy doesn't actually create a `(4, 4, 3)` copy of the adjustment array. It uses **strides** (remember those?) to virtually "repeat" the `(3,)` array across every pixel — zero extra memory.

### Real-World Use Cases

| Adjustment | `(3,)` array | Effect |
|---|---|---|
| Warm tint | `[10, 0, -10]` | More red, less blue |
| Cool tint | `[-10, 0, 10]` | More blue, less red |
| Green boost | `[0, 15, 0]` | Brighter greens (vegetation detection) |
| Mean subtraction | `[123.68, 116.78, 103.94]` | ImageNet preprocessing |

In [6]:
# --- Broadcasting in action ---

# Create a simulated image (small for readability)
img = np.zeros((4, 4, 3), dtype=np.float32)

# A per-channel brightness adjustment: shape (3,)
brightness = np.array([10, 5, 0], dtype=np.float32)

# This WORKS even though shapes don't match!
# (4, 4, 3) + (3,) → broadcasting stretches (3,) across every pixel
result = img + brightness
print("Result shape:", result.shape)  # (4, 4, 3)
print("\nResult (first row):")
print(result[0])
# Every pixel now has [10, 5, 0] added to its RGB values
# R += 10, G += 5, B += 0

Result shape: (4, 4, 3)

Result (first row):
[[10.  5.  0.]
 [10.  5.  0.]
 [10.  5.  0.]
 [10.  5.  0.]]


### Broadcasting Compatibility: Which Shape Pairs Work?

Let's systematically check which shape combinations broadcast successfully. The rule again: compare **right to left**, each dimension must be **equal** or one must be **1** (or missing).

### Compatible Pairs for `(480, 640, 3)`

```
(480, 640, 3) + (3,)        ✓  3==3, missing dims treated as 1
(480, 640, 3) + (1,)        ✓  1 broadcasts to 3 (and to 640, and to 480)
(480, 640, 3) + (640, 3)    ✓  640==640, 3==3, missing dim treated as 1
(480, 640, 3) + (480, 1, 3) ✓  480==480, 1→640, 3==3
```

### Visual: How Each Shape Broadcasts

```
(3,)         →  stretched to (1, 1, 3)  →  (480, 640, 3)  ← per-channel value
(640, 3)     →  stretched to (1, 640, 3) →  (480, 640, 3)  ← per-column value
(480, 1, 3)  →  1 expands to 640         →  (480, 640, 3)  ← per-row value
scalar       →  stretched everywhere      →  (480, 640, 3)  ← same value everywhere
```

### The "Missing Dimension" Rule

When one array has fewer dimensions, NumPy **prepends 1s** on the left:
- `(3,)` is treated as `(1, 1, 3)` when compared with `(480, 640, 3)`
- `(640, 3)` is treated as `(1, 640, 3)`

This is why `(3,)` broadcasts with `(480, 640, 3)` — the missing dimensions are filled with 1s, and 1 broadcasts to anything.

In [7]:
# --- Broadcasting compatibility examples ---

# These all WORK:
print("=== Broadcasting that SUCCEEDS ===")

# (480, 640, 3) + (3,) → OK (3 matches 3)
a = np.zeros((480, 640, 3))
b = np.array([1, 2, 3])
print(f"{a.shape} + {b.shape} → OK, result: {(a + b).shape}")

# (480, 640, 3) + (1,) → OK (1 broadcasts to anything)
c = np.array([5])
print(f"{a.shape} + {c.shape} → OK, result: {(a + c).shape}")

# (480, 640, 3) + (640, 3) → OK (640 matches 640, 3 matches 3)
d = np.zeros((640, 3))
print(f"{a.shape} + {d.shape} → OK, result: {(a + d).shape}")

# (480, 640, 3) + (480, 1, 3) → OK (1 broadcasts to 640)
e = np.zeros((480, 1, 3))
print(f"{a.shape} + {e.shape} → OK, result: {(a + e).shape}")

# (3, 4) + (4,) → OK (4 matches 4, missing dim is treated as 1)
f = np.ones((3, 4))
g = np.array([1, 2, 3, 4])
print(f"{f.shape} + {g.shape} → OK, result: {(f + g).shape}")

=== Broadcasting that SUCCEEDS ===
(480, 640, 3) + (3,) → OK, result: (480, 640, 3)
(480, 640, 3) + (1,) → OK, result: (480, 640, 3)
(480, 640, 3) + (640, 3) → OK, result: (480, 640, 3)
(480, 640, 3) + (480, 1, 3) → OK, result: (480, 640, 3)
(3, 4) + (4,) → OK, result: (3, 4)


### When Broadcasting FAILS

If the shapes don't satisfy the broadcasting rule (dimensions must be **equal** or one must be **1**), NumPy raises a `ValueError` — it does NOT guess or silently produce wrong results.

### Common Failure Cases

| Shape A | Shape B | Why it fails |
|---|---|---|
| `(480, 640, 3)` | `(5,)` | 5 ≠ 3 (rightmost dims don't match, neither is 1) |
| `(3, 4)` | `(3,)` | 3 ≠ 4 (rightmost dims: 4 vs 3) |
| `(4, 3)` | `(3, 4)` | 3 ≠ 4 AND 4 ≠ 3 (both dimensions mismatch) |

### How to Debug Broadcasting Errors

When you see `could not broadcast together`, check shapes **right to left**:

```python
print(a.shape, b.shape)   # e.g., (480, 640, 3) and (5,)
#                     3 vs 5  ← MISMATCH! This is where it breaks
```

### The Fix

Usually one of:
- **Reshape** the smaller array: `b.reshape(1, 1, 3)` to add missing dimensions
- **Fix the data**: if you have 5 values but the image has 3 channels, your data is wrong
- **Use a scalar**: if you want to add the same value to everything, use a single number, not an array

In [8]:
# --- Broadcasting that FAILS ---

print("=== Broadcasting that FAILS ===")

# (480, 640, 3) + (5,) → FAIL (5 doesn't match 3, and isn't 1)
a = np.zeros((480, 640, 3))
b = np.zeros((5,))
try:
    result = a + b
except ValueError as err:
    print(f"{a.shape} + {b.shape} → FAIL: {err}")

# (3, 4) + (3,) → FAIL (3 doesn't match 4)
c = np.ones((3, 4))
d = np.array([1, 2, 3])
try:
    result = c + d
except ValueError as err:
    print(f"{c.shape} + {d.shape} → FAIL: {err}")

# (4, 3) + (3, 4) → FAIL (3 doesn't match 4, and 4 doesn't match 3)
e = np.ones((4, 3))
f = np.ones((3, 4))
try:
    result = e + f
except ValueError as err:
    print(f"{e.shape} + {f.shape} → FAIL: {err}")

=== Broadcasting that FAILS ===
(480, 640, 3) + (5,) → FAIL: operands could not be broadcast together with shapes (480,640,3) (5,) 
(3, 4) + (3,) → FAIL: operands could not be broadcast together with shapes (3,4) (3,) 
(4, 3) + (3, 4) → FAIL: operands could not be broadcast together with shapes (4,3) (3,4) 


### Real-World Broadcasting in Edge AI Pipelines

Broadcasting isn't just a convenience — it's used in **every major image preprocessing pipeline**. Here are the three most common patterns:

### 1. Per-Channel Mean Subtraction (ImageNet Preprocessing)

Pre-trained models (ResNet, VGG, MobileNet) were trained on ImageNet, where each channel has a known mean. Before feeding an image to the model, you subtract these means:

```python
mean = np.array([123.68, 116.78, 103.94])  # R, G, B means
img_normalized = img - mean  # (224,224,3) - (3,) → broadcasts!
```

The `(3,)` mean array is broadcast across every pixel — each R value gets 123.68 subtracted, each G gets 116.78, each B gets 103.94.

### 2. Per-Channel Standardization

After subtracting the mean, you divide by per-channel standard deviation:

```python
std = np.array([58.395, 57.120, 57.375])
img_standardized = (img - mean) / std  # broadcasting handles both - and /
```

### 3. Scalar Broadcasting (Simplest Case)

Adding/subtracting a single number to every element:

```python
brighter = img + 20   # scalar 20 broadcasts to every pixel and channel
```

### Why This Is Powerful

Without broadcasting, you'd need nested loops or `np.tile()` to manually expand the small array to match the image. Broadcasting does this **virtually** — no memory is wasted on duplicated data.

In [9]:
# --- Real Edge AI broadcasting examples ---

# Example 1: Per-channel mean subtraction (ImageNet preprocessing)
# This is a standard preprocessing step for many CNN models
img = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8).astype(np.float32)

# ImageNet per-channel mean values (R, G, B)
mean = np.array([123.68, 116.78, 103.94], dtype=np.float32)

# Broadcasting subtracts the mean from EVERY pixel automatically
normalized = img - mean  # shape (224, 224, 3) - (3,) → (224, 224, 3)
print("Per-channel mean subtraction:")
print(f"  Original range: [{img.min():.1f}, {img.max():.1f}]")
print(f"  After subtraction: [{normalized.min():.1f}, {normalized.max():.1f}]")

# Example 2: Per-channel standard deviation normalization
std = np.array([58.395, 57.120, 57.375], dtype=np.float32)
standardized = (img - mean) / std  # Broadcasting both subtraction and division
print(f"\nAfter standardization: [{standardized.min():.2f}, {standardized.max():.2f}]")

# Example 3: Adding a scalar to every element (simplest broadcasting)
brighter = img + 20  # (224, 224, 3) + scalar → adds 20 to every pixel
print(f"\nAfter adding 20: [{brighter.min():.1f}, {brighter.max():.1f}]")

Per-channel mean subtraction:
  Original range: [0.0, 255.0]
  After subtraction: [-123.7, 151.1]

After standardization: [-2.12, 2.63]

After adding 20: [20.0, 275.0]


---
## 2. uint8 vs float32 — Memory and Arithmetic Implications

Images from a camera naturally come as `uint8`, but most AI models expect `float32`. Understanding why both exist, and what happens when you mix them up, is critical.

### Memory Comparison

| dtype | Bytes per element | 480×640×3 image memory |
|---|---|---|
| `uint8` | 1 byte | 921,600 bytes (~0.9 MB) |
| `float32` | 4 bytes | 3,686,400 bytes (~3.5 MB) |

→ **float32 uses 4x MORE memory** for the exact same image!

### Why uint8 for raw images
- Matches exactly what a camera sensor naturally outputs — 256 brightness levels per channel
- Uses 4x less memory and bandwidth than float32 — critical on a Raspberry Pi with limited RAM

### Why float32 for AI models
- Neural networks use continuous math (gradients, decimals) — they need values like `0.4536`, not just whole numbers 0-255
- Most models expect pixel values normalized to a range like `0.0 to 1.0`, or `-1.0 to 1.0`

In [10]:
# --- Memory comparison: uint8 vs float32 ---

# Same image, two representations
img_uint8 = np.zeros((480, 640, 3), dtype=np.uint8)
img_float32 = np.zeros((480, 640, 3), dtype=np.float32)

print("uint8 image:")
print(f"  Shape: {img_uint8.shape}")
print(f"  Elements: {img_uint8.size:,}")
print(f"  Bytes per element: {img_uint8.itemsize}")
print(f"  Total memory: {img_uint8.nbytes:,} bytes ({img_uint8.nbytes / 1024 / 1024:.2f} MB)")

print(f"\nfloat32 image:")
print(f"  Shape: {img_float32.shape}")
print(f"  Elements: {img_float32.size:,}")
print(f"  Bytes per element: {img_float32.itemsize}")
print(f"  Total memory: {img_float32.nbytes:,} bytes ({img_float32.nbytes / 1024 / 1024:.2f} MB)")

print(f"\n→ float32 uses {img_float32.nbytes / img_uint8.nbytes}x MORE memory for the same image!")
print("→ On a Raspberry Pi with 1-4 GB RAM, this 4x difference is significant when processing video streams")

uint8 image:
  Shape: (480, 640, 3)
  Elements: 921,600
  Bytes per element: 1
  Total memory: 921,600 bytes (0.88 MB)

float32 image:
  Shape: (480, 640, 3)
  Elements: 921,600
  Bytes per element: 4
  Total memory: 3,686,400 bytes (3.52 MB)

→ float32 uses 4.0x MORE memory for the same image!
→ On a Raspberry Pi with 1-4 GB RAM, this 4x difference is significant when processing video streams


### The Correct Conversion Pipeline: uint8 → float32 → Normalize

AI models expect `float32` values in a normalized range, not raw `uint8` (0–255). The conversion must happen in the right order.

### The Standard Pipeline

```
uint8 (0-255)  →  astype(float32)  →  divide by 255.0  →  float32 (0.0-1.0)
```

### Two Common Normalization Schemes

| Scheme | Formula | Output range | Used by |
|---|---|---|---|
| **0 to 1** | `img.astype(np.float32) / 255.0` | [0.0, 1.0] | PyTorch, general models |
| **-1 to 1** | `(img.astype(np.float32) - 127.5) / 127.5` | [-1.0, 1.0] | MobileNet, some TensorFlow models |
| **Mean-subtracted** | `(img - mean) / std` | ~[-2, +2] | ImageNet-pretrained models |

### Why Convert to float32 FIRST?

```
WRONG:  img_uint8 / 255.0     → uint8 can't hold 0.392 — fractional values lost!
CORRECT: img.astype(np.float32) / 255.0  → 0.392 preserved
```

`uint8` can only represent whole numbers (0, 1, 2, ..., 255). Dividing `100` by `255` should give `0.392`, but uint8 would truncate it to `0`. By converting to `float32` first, you preserve the decimal precision that neural networks need.

In [11]:
# --- The correct conversion: uint8 → float32 → normalize ---

# Raw camera data (uint8, 0-255 range)
img_uint8 = np.array([100, 200, 255], dtype=np.uint8)
print("Raw uint8 image values:", img_uint8)

# WRONG — dividing uint8 by 255 can silently misbehave
# because uint8 can't represent fractional or negative intermediate results
# wrong = img_uint8 / 255.0  # May work due to type promotion, but DON'T rely on it

# CORRECT — convert to float32 FIRST, then normalize
img_float = img_uint8.astype(np.float32) / 255.0
print("After float32 conversion + normalization:", img_float)
# [0.392, 0.784, 1.0] — values now in 0.0 to 1.0 range as models expect

# Alternative normalization to [-1.0, 1.0] range (used by some models like MobileNet)
img_normalized = (img_uint8.astype(np.float32) - 127.5) / 127.5
print("Normalized to [-1, 1] range:", img_normalized)
# [-0.216, 0.569, 1.0] — centered around 0

Raw uint8 image values: [100 200 255]
After float32 conversion + normalization: [0.39215687 0.78431374 1.        ]
Normalized to [-1, 1] range: [-0.21568628  0.5686275   1.        ]


---
## 3. The uint8 Overflow Bug — Silent and Dangerous

### ⚠️ This is one of the most dangerous bugs in image processing!

`uint8` arithmetic **wraps around silently** instead of raising an error. Your code runs fine, gives no warning, and just produces quietly wrong numbers.

```
uint8 range: 0 to 255
200 + 100 = 300 → but 300 > 255 → wraps to 300 - 256 = 44
```

**Always convert to a larger type (like `int16`, `int32`, or `float32`) before doing math that might exceed 255**, then convert back to `uint8` only at the very end if needed.

In [12]:
# --- The uint8 wraparound bug ---

a = np.array([200], dtype=np.uint8)
b = np.array([100], dtype=np.uint8)

result = a + b  # 200 + 100 = 300, but uint8 max is 255!
print(f"uint8 addition: {a[0]} + {b[0]} = {result[0]}")
print("  Expected: 300")
print("  Got: 44 ← WRAPPED AROUND (300 - 256 = 44), NOT an error!")

# More examples of uint8 overflow
x = np.array([250, 250, 250], dtype=np.uint8)
y = np.array([10, 20, 30], dtype=np.uint8)
print(f"\nuint8: {x} + {y} = {x + y}")
# [4, 214, 224] — 250+10=260→4, 250+20=270→14... wait, let's check
# Actually: 260-256=4, 270-256=14, 280-256=24
# Hmm, the result depends on NumPy version's handling — let's be explicit:
print(f"  250+10={250+10} → uint8: {(np.array([250], dtype=np.uint8) + np.array([10], dtype=np.uint8))[0]}")
print(f"  250+20={250+20} → uint8: {(np.array([250], dtype=np.uint8) + np.array([20], dtype=np.uint8))[0]}")
print(f"  250+30={250+30} → uint8: {(np.array([250], dtype=np.uint8) + np.array([30], dtype=np.uint8))[0]}")

uint8 addition: 200 + 100 = 44
  Expected: 300
  Got: 44 ← WRAPPED AROUND (300 - 256 = 44), NOT an error!

uint8: [250 250 250] + [10 20 30] = [ 4 14 24]
  250+10=260 → uint8: 4
  250+20=270 → uint8: 14
  250+30=280 → uint8: 24


### The Fix: Upcast Before Arithmetic

The solution to uint8 overflow is simple: **convert to a larger integer type before doing math**, then convert back to uint8 at the end.

### The Safe Arithmetic Pattern

```
uint8 → upcast to int16/float32 → do the math → clip to [0, 255] → cast back to uint8
```

### Why int16?

| Type | Range | Why use it? |
|---|---|---|
| `uint8` | 0 to 255 | Too small — overflows at 255 |
| `int16` | -32768 to 32767 | Safe for additions up to ~130 values, safe for subtractions (can go negative) |
| `float32` | ±3.4 × 10³⁸ | Safe for everything, needed if you want fractional results |

### Example: 200 + 100

```
uint8:   200 + 100 = 44   ← WRONG (wrapped: 300 - 256)
int16:   200 + 100 = 300  ← CORRECT
clip:    300 → 255         ← Clamped to valid uint8 range
uint8:   255               ← Final correct answer
```

### Why Clip Before Casting?

If you cast `300` (int16) directly to `uint8` without clipping, it wraps to `44` again! The clip ensures the value is within `[0, 255]` **before** the cast, so the conversion is safe.

In [13]:
# --- The correct way: upcast before arithmetic ---

a = np.array([200], dtype=np.uint8)
b = np.array([100], dtype=np.uint8)

# WRONG: direct uint8 addition wraps around
wrong = a + b
print(f"WRONG (uint8 + uint8): {a[0]} + {b[0]} = {wrong[0]}")

# CORRECT: upcast to int16 (or larger) before arithmetic
correct_int16 = a.astype(np.int16) + b.astype(np.int16)
print(f"CORRECT (int16): {a[0]} + {b[0]} = {correct_int16[0]}")

# CORRECT: upcast to float32 (if you need fractional results too)
correct_f32 = a.astype(np.float32) + b.astype(np.float32)
print(f"CORRECT (float32): {a[0]} + {b[0]} = {correct_f32[0]}")

# Convert back to uint8 at the end (with clipping to prevent overflow)
final = np.clip(correct_int16, 0, 255).astype(np.uint8)
print(f"\nFinal (clipped back to uint8): {final[0]}")
# 300 gets clipped to 255 — the maximum uint8 value

WRONG (uint8 + uint8): 200 + 100 = 44
CORRECT (int16): 200 + 100 = 300
CORRECT (float32): 200 + 100 = 300.0

Final (clipped back to uint8): 255


### Practical Safe Brightness Adjustment

Putting it all together — the **safe arithmetic pattern** for any pixel math on uint8 images:

```
np.clip(img.astype(np.int16) + value, 0, 255).astype(np.uint8)
```

### Why Each Step Is Necessary

| Step | Without it | What goes wrong |
|---|---|---|
| `.astype(np.int16)` | `img + value` on uint8 | Overflow wraps around: 230 + 30 → 4 (not 260) |
| `np.clip(..., 0, 255)` | `int16` result may exceed 255 | Values like 260 would overflow when cast back to uint8 |
| `.astype(np.uint8)` | Result stays int16 | 2x unnecessary memory, model expects uint8 |

### The Bug in Action

A bright pixel `[230, 240, 250]` + 30:
- **Wrong** (uint8): `[4, 14, 24]` — wrapped around to DARK values! The image gets darker where it should be brighter.
- **Correct** (safe): `[255, 255, 255]` — clipped to maximum brightness. Makes sense!

### Why This Pattern Appears Everywhere

This same `upcast → math → clip → cast back` pattern is used for:
- Brightness/contrast adjustment
- Image blending (alpha compositing)
- Filter convolutions (sum of weighted neighbors)
- Any arithmetic that could exceed the 0–255 range

In [14]:
# --- Practical: safe brightness adjustment ---

# Simulated image
img = np.random.randint(0, 256, (4, 4, 3), dtype=np.uint8)
print("Original image (first pixel):", img[0, 0])

# WRONG: adding 30 directly to uint8 can wrap around for bright pixels
# If a pixel is 230 and we add 30: 230 + 30 = 260 → wraps to 4 (dark!)
wrong = img + 30
print("\nWRONG (uint8 + 30, wraps): first pixel:", wrong[0, 0])

# CORRECT: upcast to int16, add, clip, convert back
safe = np.clip(img.astype(np.int16) + 30, 0, 255).astype(np.uint8)
print("CORRECT (int16 + 30, clipped): first pixel:", safe[0, 0])

# The difference is visible for bright pixels
bright_pixel = np.array([230, 240, 250], dtype=np.uint8)
print(f"\nBright pixel {bright_pixel}:")
print(f"  WRONG (+30 uint8): {bright_pixel + np.uint8(30)} → wrapped to dark values!")
print(f"  CORRECT (+30 safe): {np.clip(bright_pixel.astype(np.int16) + 30, 0, 255).astype(np.uint8)} → clipped to 255")

Original image (first pixel): [117 225  47]

WRONG (uint8 + 30, wraps): first pixel: [147 255  77]
CORRECT (int16 + 30, clipped): first pixel: [147 255  77]

Bright pixel [230 240 250]:
  WRONG (+30 uint8): [ 4 14 24] → wrapped to dark values!
  CORRECT (+30 safe): [255 255 255] → clipped to 255


---
## 4. Aggregation Operations

NumPy provides fast, built-in aggregation functions that operate on entire arrays at once. These are essential for image statistics, quality checks, and feature extraction in Edge AI pipelines.

### Aggregation Operations: Computing Statistics Over Entire Arrays

Aggregations reduce an array to a **single summary value** (or a few values) — like mean, sum, min, max, or standard deviation. NumPy computes these in optimized C code, making them **100x+ faster** than Python loops.

### Global vs Per-Axis Aggregation

```
Global (entire image):     img.mean()          → single number (average brightness)
Per-channel:               img.mean(axis=(0,1)) → shape (3,)  (R mean, G mean, B mean)
```

### Common Aggregations in Edge AI

| Function | What it computes | Image processing use |
|---|---|---|
| `img.mean()` | Average pixel value | Overall brightness |
| `img.std()` | Standard deviation | Contrast / dynamic range |
| `img.min()` | Darkest pixel | Check for crushed blacks |
| `img.max()` | Brightest pixel | Check for blown highlights |
| `img.sum()` | Total of all pixels | Weighted area calculations |

### The `axis` Parameter

The `axis` parameter specifies **which dimension to collapse**:
- `axis=0` → collapse rows (aggregate down each column)
- `axis=1` → collapse columns (aggregate across each row)
- `axis=2` → collapse channels (convert color → grayscale)
- `axis=(0, 1)` → collapse both rows AND columns → one value per channel

For a `(480, 640, 3)` image: `img.mean(axis=(0, 1))` gives shape `(3,)` — the mean R, G, B values across the entire image.

In [15]:
# --- Basic aggregations ---

img = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

# Global statistics
print("Global image statistics:")
print(f"  Mean:   {img.mean():.2f}")
print(f"  Std:    {img.std():.2f}")
print(f"  Min:    {img.min()}")
print(f"  Max:    {img.max()}")
print(f"  Sum:    {img.sum()}")

# Per-channel statistics using axis parameter
# axis=(0, 1) means aggregate over both height and width → result is per-channel
channel_means = img.mean(axis=(0, 1))
print(f"\nPer-channel mean (R, G, B): {channel_means}")
print(f"  Shape: {channel_means.shape}")  # (3,) — one mean per channel

channel_stds = img.std(axis=(0, 1))
print(f"Per-channel std:  {channel_stds}")

channel_mins = img.min(axis=(0, 1))
channel_maxs = img.max(axis=(0, 1))
print(f"Per-channel min:  {channel_mins}")
print(f"Per-channel max:  {channel_maxs}")

Global image statistics:
  Mean:   127.62
  Std:    73.93
  Min:    0
  Max:    255
  Sum:    117614165

Per-channel mean (R, G, B): [127.57193685 127.61403646 127.67263672]
  Shape: (3,)
Per-channel std:  [74.01069787 73.91507367 73.87481463]
Per-channel min:  [0 0 0]
Per-channel max:  [255 255 255]


### Aggregation Along Specific Axes: The `axis` Parameter

The `axis` parameter controls **which dimension(s) to collapse**. The result loses those dimensions and keeps the rest.

### How `axis` Works on a 3D Image `(H, W, C)`

```
Image: (4, 5, 3)  — 4 rows, 5 columns, 3 channels

axis=0 → collapse rows    → result shape (5, 3)   — one value per column per channel
axis=1 → collapse columns  → result shape (4, 3)   — one value per row per channel
axis=2 → collapse channels → result shape (4, 5)   — one value per pixel = grayscale!
```

### Visual: Collapsing Each Axis

```
axis=0 (average over rows):        axis=2 (average over channels):
┌───┬───┬───┬───┬───┐              ┌───┬───┬───┬───┬───┐
│R1 │   │   │   │   │              │p00│p01│p02│p03│p04│  → avg(R,G,B) per pixel
├───┤   │   │   │   │              ├───┼───┼───┼───┼───┤
│R2 │   │   │   │   │  → avg       │p10│p11│p12│p13│p14│  → avg(R,G,B) per pixel
├───┤   │   │   │   │  downward    ├───┼───┼───┼───┼───┤
│R3 │   │   │   │   │              │p20│p21│p22│p23│p24│
├───┤   │   │   │   │              ├───┼───┼───┼───┼───┤
│R4 │   │   │   │   │              │p30│p31│p32│p33│p34│
└───┴───┴───┴───┴───┘              └───┴───┴───┴───┴───┘
Result: (5, 3)                      Result: (4, 5) = grayscale image!
```

### The Key Insight: Grayscale Conversion

`img.mean(axis=2)` is the simplest way to convert a color image to grayscale — it averages the R, G, B channels for each pixel into a single brightness value.

### `argmin` / `argmax`: Finding Positions

These return the **index** of the minimum or maximum value, not the value itself. Useful for finding the brightest pixel's location or the most confident class prediction.

In [16]:
# --- Aggregation along specific axes ---

img = np.random.randint(0, 256, (4, 5, 3), dtype=np.uint8)
print("Image shape:", img.shape)

# Mean along axis 0 (average over rows/height) → one value per (column, channel)
mean_over_rows = img.mean(axis=0)
print(f"\nMean over axis 0 (rows): shape={mean_over_rows.shape}")  # (5, 3)

# Mean along axis 1 (average over columns/width) → one value per (row, channel)
mean_over_cols = img.mean(axis=1)
print(f"Mean over axis 1 (cols): shape={mean_over_cols.shape}")  # (4, 3)

# Mean along axis 2 (average over channels) → grayscale conversion!
grayscale = img.mean(axis=2)
print(f"Mean over axis 2 (channels): shape={grayscale.shape}")  # (4, 5)
print("This is a simple grayscale conversion!")

# argmin / argmax — find the index of the min/max value
flat = np.array([3, 1, 4, 1, 5, 9, 2, 6])
print(f"\nArray: {flat}")
print(f"  Index of max: {flat.argmax()} (value: {flat.max()})")
print(f"  Index of min: {flat.argmin()} (value: {flat.min()})")

Image shape: (4, 5, 3)

Mean over axis 0 (rows): shape=(5, 3)
Mean over axis 1 (cols): shape=(4, 3)
Mean over axis 2 (channels): shape=(4, 5)
This is a simple grayscale conversion!

Array: [3 1 4 1 5 9 2 6]
  Index of max: 5 (value: 9)
  Index of min: 1 (value: 1)


### Practical Application: Image Quality Assessment

Aggregations are commonly used to automatically assess image quality **before** feeding images to an AI model. Bad inputs produce bad predictions — so you check first.

### Quality Indicators from Aggregations

| Metric | What it tells you | Healthy range (uint8) |
|---|---|---|
| `img.mean()` | Overall brightness | ~100–160 (not too dark/bright) |
| `img.std()` | Contrast / dynamic range | >40 (low std = flat, featureless image) |
| `img.max()` | Brightest pixel | <255 (255 = blown out / saturated) |
| `img.min()` | Darkest pixel | >0 (0 = crushed blacks) |

### Common Problems Detected

```
Underexposed (too dark):     mean ≈ 20,  max ≈ 50
Overexposed (too bright):    mean ≈ 240, min ≈ 200
Low contrast (flat):         std ≈ 5
Well-exposed:                mean ≈ 128, std ≈ 50
```

### Why This Matters in Edge AI

On an edge device (Raspberry Pi, Jetson), you might be capturing frames from a camera in varying lighting conditions. Running an AI model on a pitch-black frame wastes compute and gives garbage results. A quick `img.mean() < 30` check lets you **skip processing** for bad frames — saving power and time.

In [17]:
# --- Practical: checking image quality with aggregations ---

# Simulate a dark image (most pixels near 0)
dark_img = np.random.randint(0, 50, (480, 640, 3), dtype=np.uint8)
print("Dark image:")
print(f"  Mean brightness: {dark_img.mean():.1f} (low → underexposed)")
print(f"  Max pixel value: {dark_img.max()}")

# Simulate a bright/blown-out image (most pixels near 255)
bright_img = np.random.randint(200, 256, (480, 640, 3), dtype=np.uint8)
print(f"\nBright image:")
print(f"  Mean brightness: {bright_img.mean():.1f} (high → overexposed)")
print(f"  Max pixel value: {bright_img.max()}")

# Well-exposed image
good_img = np.random.randint(50, 200, (480, 640, 3), dtype=np.uint8)
print(f"\nWell-exposed image:")
print(f"  Mean brightness: {good_img.mean():.1f} (good range)")
print(f"  Std (contrast):  {good_img.std():.1f} (higher = more contrast)")

Dark image:
  Mean brightness: 24.5 (low → underexposed)
  Max pixel value: 49

Bright image:
  Mean brightness: 227.5 (high → overexposed)
  Max pixel value: 255

Well-exposed image:
  Mean brightness: 124.5 (good range)
  Std (contrast):  43.3 (higher = more contrast)


---
## 5. Boolean Masking and Conditional Operations

Boolean masking lets you select and modify array elements based on conditions — no loops needed. This is how you do thresholding, masking, and conditional pixel operations in Edge AI.

### Boolean Masking: Selecting Elements by Condition

A **boolean mask** is an array of `True`/`False` values that acts as a filter — you "overlay" it on another array to select only the elements where the mask is `True`.

### The Concept

```
Original array:  [10,  25,  50,  75,  100, 150, 200]
Condition:       [10>50, 25>50, 50>50, 75>50, 100>50, 150>50, 200>50]
Mask:            [F,    F,     F,     T,     T,      T,      T    ]
Selected:        [75, 100, 150, 200]  ← only elements where mask is True
```

### Combining Conditions

You can combine multiple conditions using **bitwise operators** (not `and`/`or`):

| Operator | Meaning | Example |
|---|---|---|
| `&` | AND (both must be True) | `(arr > 30) & (arr < 150)` |
| `\|` | OR (either can be True) | `(arr < 20) \| (arr > 180)` |
| `~` | NOT (invert) | `~(arr > 50)` = `arr <= 50` |

⚠️ **Always use parentheses** around each condition: `(arr > 30) & (arr < 150)`, not `arr > 30 & arr < 150` (operator precedence will bite you).

### Modifying with Masks

```python
arr[arr > 100] = 100   # cap all values above 100 to 100
```

This is vectorized — it modifies **every matching element at once**, no loop needed.

In [18]:
# --- Boolean masking basics ---

arr = np.array([10, 25, 50, 75, 100, 150, 200])

# Create a boolean mask (True/False array)
mask = arr > 50
print("Array:", arr)
print("Mask (arr > 50):", mask)

# Use the mask to select elements
selected = arr[mask]
print("Selected (arr[mask]):", selected)  # [75, 100, 150, 200]

# Combine conditions with & (and), | (or)
mask2 = (arr > 30) & (arr < 150)
print("\nMask (30 < arr < 150):", mask2)
print("Selected:", arr[mask2])  # [50, 75, 100]

# Modify elements based on a mask
arr_copy = arr.copy()
arr_copy[arr_copy > 100] = 100  # Clip all values above 100 to 100
print("\nAfter clipping >100 to 100:", arr_copy)

Array: [ 10  25  50  75 100 150 200]
Mask (arr > 50): [False False False  True  True  True  True]
Selected (arr[mask]): [ 75 100 150 200]

Mask (30 < arr < 150): [False False  True  True  True False False]
Selected: [ 50  75 100]

After clipping >100 to 100: [ 10  25  50  75 100 100 100]


### Image Thresholding with Boolean Masks

Thresholding converts a grayscale image into a **binary image** — every pixel is either "on" (255) or "off" (0) based on a brightness cutoff. This is a fundamental step in many classical vision pipelines (edge detection, object segmentation, motion detection).

### How It Works

```
Original:    [ 10, 50, 128, 200, 255, 30, 180 ]
Mask (>128): [False, False, True, True, True, False, True]
Binary:      [  0,   0, 255, 255, 255,   0, 255 ]
```

### Counting Pixels with `np.sum`

A boolean array has `True=1` and `False=0`, so `np.sum(mask)` counts how many pixels satisfy the condition — **no loop needed**:

```python
bright_count = np.sum(img > 200)   # counts all pixels above 200 at once
```

### Real-World Examples

| Use case | Condition |
|---|---|
| Detect bright spots (LEDs, headlights) | `img > 200` |
| Detect dark regions (shadows, defects) | `img < 50` |
| Mid-range extraction (skin tone in grayscale) | `(img > 80) & (img < 200)` |
| Percentage of bright pixels | `np.sum(img > 128) / img.size * 100` |

In [19]:
# --- Image thresholding with boolean masks ---

# Create a simulated image
img = np.random.randint(0, 256, (8, 8), dtype=np.uint8)
print("Original image:")
print(img)

# Threshold: create a binary mask where pixels > 128 are True
mask = img > 128
print("\nThreshold mask (> 128):")
print(mask.astype(int))  # Show as 0/1 for readability

# Create a binary image (0 or 255)
binary = np.where(img > 128, 255, 0).astype(np.uint8)
print("\nBinary image (threshold at 128):")
print(binary)

# Count pixels above a threshold — no loop needed!
bright_count = np.sum(img > 200)
total_pixels = img.size
print(f"\nPixels > 200: {bright_count} out of {total_pixels} ({bright_count/total_pixels*100:.1f}%)")

Original image:
[[ 81 187 165  95 216  42  13 222]
 [247 129  18 134  56 239 240  87]
 [101 111 244   4  92 202 143 234]
 [220  74 185  13  93 101  32 120]
 [  9 133 167 208  89  85 178   0]
 [239 215 125 225 255  66 201 204]
 [239 186 118 181 156 109  48 120]
 [ 15  16 190 140  94  50  33 187]]

Threshold mask (> 128):
[[0 1 1 0 1 0 0 1]
 [1 1 0 1 0 1 1 0]
 [0 0 1 0 0 1 1 1]
 [1 0 1 0 0 0 0 0]
 [0 1 1 1 0 0 1 0]
 [1 1 0 1 1 0 1 1]
 [1 1 0 1 1 0 0 0]
 [0 0 1 1 0 0 0 1]]

Binary image (threshold at 128):
[[  0 255 255   0 255   0   0 255]
 [255 255   0 255   0 255 255   0]
 [  0   0 255   0   0 255 255 255]
 [255   0 255   0   0   0   0   0]
 [  0 255 255 255   0   0 255   0]
 [255 255   0 255 255   0 255 255]
 [255 255   0 255 255   0   0   0]
 [  0   0 255 255   0   0   0 255]]

Pixels > 200: 17 out of 64 (26.6%)


### `np.where`: Conditional Element Selection

`np.where(condition, x, y)` is a vectorized if-else: for every element, it picks from `x` if the condition is `True`, or from `y` if `False`.

```
condition:  [True,  False, True,  True,  False]
x:          [255,  255,   255,   255,   255]
y:          [0,    0,     0,     0,     0]
result:     [255,  0,     255,   255,   0]    ← picks x where True, y where False
```

### Common Use Cases in Image Processing

| Pattern | What it does |
|---|---|
| `np.where(img > 128, 255, 0)` | **Binarization** — bright pixels → 255, dark → 0 |
| `np.where(img < 64, 0, np.where(img < 192, 128, 255))` | **Multi-level segmentation** — 3 intensity bands |
| `np.where(mask, img, 0)` | **Masking** — keep pixels where mask is True, zero elsewhere |

### `np.where` vs `np.clip`

- `np.where` gives you **full control** — different values for different conditions
- `np.clip(arr, min, max)` is simpler when you just need to **clamp** values to a range
- Use `np.where` for thresholding/segmentation, `np.clip` for preventing overflow

In [20]:
# --- np.where: conditional array creation ---

# np.where(condition, x, y) → elements from x where condition is True, from y otherwise
img = np.random.randint(0, 256, (4, 4), dtype=np.uint8)
print("Original:")
print(img)

# Apply a threshold: pixels > 128 become 255, others become 0
thresholded = np.where(img > 128, 255, 0).astype(np.uint8)
print("\nThresholded (>128 → 255, else → 0):")
print(thresholded)

# More complex: highlight mid-range pixels
# Dark pixels → 0, mid-range → 128, bright → 255
segmented = np.where(img < 64, 0, np.where(img < 192, 128, 255)).astype(np.uint8)
print("\nSegmented (3-level):")
print(segmented)

# np.clip — a safer alternative for simple clamping
clipped = np.clip(img, 50, 200)
print("\nClipped to [50, 200]:")
print(clipped)

Original:
[[ 66  42 244 251]
 [ 65 100 239  67]
 [138  59  10  54]
 [113  55  99 128]]

Thresholded (>128 → 255, else → 0):
[[  0   0 255 255]
 [  0   0 255   0]
 [255   0   0   0]
 [  0   0   0   0]]

Segmented (3-level):
[[128   0 255 255]
 [128 128 255 128]
 [128   0   0   0]
 [128   0 128 128]]

Clipped to [50, 200]:
[[ 66  50 200 200]
 [ 65 100 200  67]
 [138  59  50  54]
 [113  55  99 128]]


### Fancy Indexing: Selecting Elements by Index Array

Fancy indexing lets you select **multiple specific elements** at once using a list or array of indices — like picking items from a shelf by their position numbers.

### Regular vs Fancy Indexing

```python
# Regular indexing: one element at a time
val = arr[3]

# Fancy indexing: multiple elements at once
vals = arr[[0, 3, 5, 7]]   # picks elements at positions 0, 3, 5, 7
```

### 2D Fancy Indexing

For a 2D array, you provide two index arrays — one for rows, one for columns:

```python
img[row_idx, col_idx]  # picks one element per (row, col) pair
```

Example: `img[[0, 1, 2], [3, 1, 4]]` picks:
- `img[0, 3]`, `img[1, 1]`, `img[2, 4]` — **three elements**, not a 3×3 grid

### ⚠️ Fancy Indexing Always Returns a COPY

Unlike slicing (which returns a view), fancy indexing **always copies data**. This is because the selected elements may not be contiguous or stridable in memory, so NumPy can't just adjust strides — it has to physically gather the values into a new array.

```python
view = img[0:2]              # slicing → view (shares memory)
copy = img[[0, 1]]           # fancy indexing → copy (independent)
```

In [21]:
# --- Fancy indexing: selecting with index arrays ---

arr = np.array([10, 20, 30, 40, 50, 60, 70, 80])

# Select specific elements by index
indices = [0, 3, 5, 7]
print("Array:", arr)
print(f"Elements at indices {indices}:", arr[indices])  # [10, 40, 60, 80]

# 2D fancy indexing
img = np.arange(20).reshape(4, 5)
print("\n2D array:")
print(img)

# Select specific rows
rows = [0, 2, 3]
print(f"\nRows {rows}:")
print(img[rows])

# Select specific elements using row and column index arrays
row_idx = np.array([0, 1, 2, 3])
col_idx = np.array([1, 3, 0, 4])
print(f"\nElements at (rows=[0,1,2,3], cols=[1,3,0,4]):", img[row_idx, col_idx])
# [1, 8, 10, 19] — one element per (row, col) pair

# Fancy indexing always returns a COPY, not a view!
subset = img[[0, 1]]
print(f"\nFancy indexing shares memory? {np.shares_memory(img, subset)}")  # False

Array: [10 20 30 40 50 60 70 80]
Elements at indices [0, 3, 5, 7]: [10 40 60 80]

2D array:
[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]

Rows [0, 2, 3]:
[[ 0  1  2  3  4]
 [10 11 12 13 14]
 [15 16 17 18 19]]

Elements at (rows=[0,1,2,3], cols=[1,3,0,4]): [ 1  8 10 19]

Fancy indexing shares memory? False


---
## 6. Array Concatenation and Stacking

Combining arrays is common when building batch inputs for AI models or stitching image regions together.

### Concatenation: Joining Arrays Along an Existing Axis

Concatenation means **gluing arrays together** end-to-end along one of their existing dimensions. The arrays must have the **same shape** along all axes **except** the one you're concatenating along.

### Vertical vs Horizontal Stacking

For two 2D arrays `a` and `b` (both shape `(2, 2)`):

```
axis=0 (vertical — add rows):     axis=1 (horizontal — add columns):
┌───┬───┐                         ┌───┬───┬───┬───┐
│ a │   │                         │ a │   │ b │   │
├───┤   │                         ├───┤   ├───┤   │
│   │   │                         │   │   │   │   │
├───┼───┤                         └───┴───┴───┴───┘
│ b │   │
├───┤   │
│   │   │
└───┴───┘
Result shape: (4, 2)              Result shape: (2, 4)
```

### Convenience Functions

| Function | Equivalent to | Use case |
|---|---|---|
| `np.vstack([a, b])` | `np.concatenate([a, b], axis=0)` | Stack images top-to-bottom |
| `np.hstack([a, b])` | `np.concatenate([a, b], axis=1)` | Stack images side-by-side |

### For Images

- **Vertical stack** (`axis=0`): Stitch images top-to-bottom — both must have the same width
- **Horizontal stack** (`axis=1`): Stitch images side-by-side — both must have the same height

In [22]:
# --- Concatenation and stacking ---

a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

# np.concatenate — join along an existing axis
# axis=0: stack vertically (add rows)
vstack = np.concatenate([a, b], axis=0)
print("Vertical concatenate (axis=0):")
print(vstack)

# axis=1: stack horizontally (add columns)
hstack = np.concatenate([a, b], axis=1)
print("\nHorizontal concatenate (axis=1):")
print(hstack)

# Convenience functions
print("\nnp.vstack:")
print(np.vstack([a, b]))  # Same as concatenate axis=0

print("\nnp.hstack:")
print(np.hstack([a, b]))  # Same as concatenate axis=1

Vertical concatenate (axis=0):
[[1 2]
 [3 4]
 [5 6]
 [7 8]]

Horizontal concatenate (axis=1):
[[1 2 5 6]
 [3 4 7 8]]

np.vstack:
[[1 2]
 [3 4]
 [5 6]
 [7 8]]

np.hstack:
[[1 2 5 6]
 [3 4 7 8]]


### Building Batches for AI Inference

AI models don't process one image at a time — they process **batches**. A batch is a collection of images stacked together into a single 4D array:

```
Single image:  (224, 224, 3)    → height, width, channels
Batch of N:    (N, 224, 224, 3) → batch, height, width, channels
```

### `np.stack` vs `np.concatenate`

| Function | What it does | New axis? |
|---|---|---|
| `np.stack([a, b, c], axis=0)` | Creates a **new dimension** and places arrays along it | Yes — adds a new axis |
| `np.concatenate([a, b], axis=0)` | Joins along an **existing** dimension | No — uses an existing axis |

**Use `np.stack`** when you have separate images and want to combine them into a batch.
**Use `np.concatenate`** when you want to extend an array along a dimension that already exists (e.g., stitching image tiles).

### Visual: How `np.stack` Works

```
img1: (224, 224, 3)     np.stack([img1, img2, img3], axis=0)
img2: (224, 224, 3)     ──────────────────────────────►
img3: (224, 224, 3)     batch: (3, 224, 224, 3)
                         ┌─────── batch dimension (new!) ───────┐
                         │ img1  │ img2  │ img3                  │
                         └───────┴───────┴─────── ... ───────────┘
```

In [23]:
# --- Building a batch of images for AI inference ---

# Simulate 3 individual images (224x224x3 each, like ImageNet inputs)
img1 = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
img2 = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
img3 = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)

# Create a batch by stacking along a new axis (axis=0)
# Result shape: (3, 224, 224, 3) — (batch, height, width, channels)
batch = np.stack([img1, img2, img3], axis=0)
print("Batch shape:", batch.shape)  # (3, 224, 224, 3)
print("This is the standard input format for most AI models!")

# np.stack creates a NEW axis, unlike concatenate which uses an existing one
# Compare: stacking 1D arrays into 2D
x = np.array([1, 2, 3])
y = np.array([4, 5, 6])
print("\nnp.stack([x, y], axis=0):")
print(np.stack([x, y], axis=0))  # [[1,2,3], [4,5,6]] — shape (2, 3)
print("np.stack([x, y], axis=1):")
print(np.stack([x, y], axis=1))  # [[1,4], [2,5], [3,6]] — shape (3, 2)

Batch shape: (3, 224, 224, 3)
This is the standard input format for most AI models!

np.stack([x, y], axis=0):
[[1 2 3]
 [4 5 6]]
np.stack([x, y], axis=1):
[[1 4]
 [2 5]
 [3 6]]


---
## Quick Recap

1. **Broadcasting** lets NumPy do math between different shapes by comparing dimensions right-to-left, matching if equal or one is `1`.
2. **uint8** saves memory (1 byte vs 4 bytes for float32) and matches raw camera data, but **silently wraps around on overflow** — always upcast before risky math.
3. **Safe arithmetic pattern**: `np.clip(img.astype(np.int16) + value, 0, 255).astype(np.uint8)`
4. **Aggregations** (mean, sum, min, max, std) with `axis` parameter give per-channel or per-row statistics instantly.
5. **Boolean masking** enables thresholding and conditional pixel operations without loops.
6. **np.stack** creates batches by adding a new axis — essential for AI model inputs.

## Self-Check Questions

1. Explain, in your own words, why `(480, 640, 3) + (3,)` is allowed by broadcasting, but `(480, 640, 3) + (5,)` is not.
2. A student writes `pixel_sum = a + b` where both `a` and `b` are uint8 arrays, and gets an unexpectedly small result. What's happening, and how should they fix their code?
3. How would you normalize an image from uint8 (0-255) to float32 (0.0-1.0) safely? Why must you convert dtype first?
4. Write code to count how many pixels in an image are brighter than 200, without using any loop.
5. What's the difference between `np.concatenate` and `np.stack`? When would you use each?